# control_mode_multi_test-多模式控制测试

### 初始化

In [ ]:
from easy_arm_control.control import RoboticArmAgent, ChooseGripper
import rospy
import math

# ROS节点初始化
NODE_NAME = "airbot_play_control_example"
rospy.init_node(NODE_NAME)
other_config = None
sim_player_nums = rospy.get_param("/airbot_play/sim_player_nums", default=0)
sim_type = ["real", "gazebo"][sim_player_nums]
print("Use", sim_type)
gripper = ["gripper", "inspire"][0]
if sim_type == "gazebo":
    other_config = ("", "airbot_play_arm")
gripper_controller = ChooseGripper(sim_type, gripper)()
arm = RoboticArmAgent(
    node_name=NODE_NAME, gripper=(4, gripper_controller), other_config=other_config
)

### 始终（全局）无规划、无插值快速执行（直接发送终点关节目标角度值和速度）

In [ ]:
arm.immediate_excute(0), arm.set_plan_mode(0)  # 模式设置
arm.set_move_speed(6)  # 电机速度设置
arm.joints_speed_limit(min_=2.593, max_=2 * math.pi)  # 关节速度限制
arm.go_to_named_or_joint_target({0: math.pi / 4, 5: math.pi / 2})  # 目标设置与执行

### 始终（全局）有规划、有插值正常执行（通常模式）

In [ ]:
arm.immediate_excute(-1), arm.set_plan_mode(-1)  # 模式设置
arm.set_vel_acc_max_factor(v_m=1.0, a_m=1.0)  # 速度、加速度系数
arm.joints_speed_limit(min_=2.593, max_=2 * math.pi)  # 电机速度限制
arm.go_to_named_or_joint_target({0: math.pi / 4, 5: math.pi / 2})  # 目标设置与执行

### 始终（全局）无规划、有插值执行（对逆解结果进行插值）

In [ ]:
arm.set_plan_mode(0), arm.set_interpolate_mode(-1)  # 模式设置
arm.set_move_speed(6)  # 电机速度设置
arm.joints_speed_limit(min_=2.593, max_=2 * math.pi)  # 电机速度限制
arm.go_to_named_or_joint_target({0: math.pi / 4, 5: math.pi / 2})  # 目标设置与执行

### 始终（全局）有规划、无插值执行

In [ ]:
arm.set_plan_mode(-1), arm.set_interpolate_mode(0)  # 模式设置
arm.set_move_speed(6)  # 电机速度设置
arm.joints_speed_limit(min_=2.593, max_=2 * math.pi)  # 电机速度限制
arm.go_to_named_or_joint_target({0: math.pi / 4, 5: math.pi / 2})  # 目标设置与执行

### 回到Home位置

In [ ]:
arm.immediate_excute(-1), arm.set_plan_mode(-1)
arm.go_to_named_or_joint_target("Standby", sleep_time=2)

### 退出

In [ ]:
# 关闭当前控制节点
arm.go_to_named_or_joint_target("Home", sleep_time=3)
rospy.signal_shutdown("end")